## Importing Pyspark and creating an instant of Spark


In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('rheza').getOrCreate()

## Understanding the Dataset

In [3]:
drug_trial = spark.read.json('dataset.json', multiLine= True)

In [4]:
drug_trial.show(5)

+----------------+--------------------+---------+-----------------+-------------------+------------------------------+--------------------+
|ageofparticipant|           clinician|drug_used|experimentenddate|experimentstartdate|noofhourspassedatfirstreaction|              result|
+----------------+--------------------+---------+-----------------+-------------------+------------------------------+--------------------+
|              19|{Ontario, Saul, t...|  Placebo|    1619827200000|      1617235200000|                            52|{BP normalized, r...|
|              14|{Ontario, Saul, n...| Naproxen|    1619827200000|      1617235200000|                            78|    {Follow-up, N/A}|
|              17|{Ontario, Saul, n...|  Placebo|    1619827200000|      1617235200000|                            14|    {Follow-up, N/A}|
|              18|{Ontario, Will, n...| Naproxen|    1619827200000|      1617235200000|                            14|{BP normalized, N/A}|
|              17|{O

In [5]:
drug_trial.printSchema()

root
 |-- ageofparticipant: long (nullable = true)
 |-- clinician: struct (nullable = true)
 |    |-- branch: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- role: string (nullable = true)
 |-- drug_used: string (nullable = true)
 |-- experimentenddate: string (nullable = true)
 |-- experimentstartdate: string (nullable = true)
 |-- noofhourspassedatfirstreaction: long (nullable = true)
 |-- result: struct (nullable = true)
 |    |-- conclusion: string (nullable = true)
 |    |-- sideeffectsonparticipant: string (nullable = true)



In [6]:
drug_trial.dtypes

[('ageofparticipant', 'bigint'),
 ('clinician', 'struct<branch:string,name:string,role:string>'),
 ('drug_used', 'string'),
 ('experimentenddate', 'string'),
 ('experimentstartdate', 'string'),
 ('noofhourspassedatfirstreaction', 'bigint'),
 ('result', 'struct<conclusion:string,sideeffectsonparticipant:string>')]

In [7]:
drug_trial.columns

['ageofparticipant',
 'clinician',
 'drug_used',
 'experimentenddate',
 'experimentstartdate',
 'noofhourspassedatfirstreaction',
 'result']

In [8]:
columns = ['ageofparticipant',
 'clinician.branch',
 'clinician.name',
 'clinician.role',
 'drug_used',
 'experimentenddate',
 'experimentstartdate',
 'noofhourspassedatfirstreaction',
 'result.conclusion',
 'result.sideeffectsonparticipant']

In [9]:
len(columns)

10

In [10]:


flatten_drug_trial = drug_trial.select(columns)

In [11]:
flatten_drug_trial.show(5)

+----------------+-------+-------+---------+---------+-----------------+-------------------+------------------------------+-------------+------------------------+
|ageofparticipant| branch|   name|     role|drug_used|experimentenddate|experimentstartdate|noofhourspassedatfirstreaction|   conclusion|sideeffectsonparticipant|
+----------------+-------+-------+---------+---------+-----------------+-------------------+------------------------------+-------------+------------------------+
|              19|Ontario|   Saul|therapist|  Placebo|    1619827200000|      1617235200000|                            52|BP normalized|          rashes on neck|
|              14|Ontario|   Saul|    nurse| Naproxen|    1619827200000|      1617235200000|                            78|    Follow-up|                     N/A|
|              17|Ontario|   Saul|    nurse|  Placebo|    1619827200000|      1617235200000|                            14|    Follow-up|                     N/A|
|              18|Onta

In [12]:
flatten_drug_trial.printSchema()

root
 |-- ageofparticipant: long (nullable = true)
 |-- branch: string (nullable = true)
 |-- name: string (nullable = true)
 |-- role: string (nullable = true)
 |-- drug_used: string (nullable = true)
 |-- experimentenddate: string (nullable = true)
 |-- experimentstartdate: string (nullable = true)
 |-- noofhourspassedatfirstreaction: long (nullable = true)
 |-- conclusion: string (nullable = true)
 |-- sideeffectsonparticipant: string (nullable = true)



In [13]:
drug_trial.show(5)

+----------------+--------------------+---------+-----------------+-------------------+------------------------------+--------------------+
|ageofparticipant|           clinician|drug_used|experimentenddate|experimentstartdate|noofhourspassedatfirstreaction|              result|
+----------------+--------------------+---------+-----------------+-------------------+------------------------------+--------------------+
|              19|{Ontario, Saul, t...|  Placebo|    1619827200000|      1617235200000|                            52|{BP normalized, r...|
|              14|{Ontario, Saul, n...| Naproxen|    1619827200000|      1617235200000|                            78|    {Follow-up, N/A}|
|              17|{Ontario, Saul, n...|  Placebo|    1619827200000|      1617235200000|                            14|    {Follow-up, N/A}|
|              18|{Ontario, Will, n...| Naproxen|    1619827200000|      1617235200000|                            14|{BP normalized, N/A}|
|              17|{O

In [14]:
from pyspark.sql import functions as fn

In [17]:
drug_trial.select([fn.count(fn.when(fn.col(column).isNull(),column)).alias(column) for column in columns]).show() #checking the nulls

+----------------+----------------+--------------+--------------+---------+-----------------+-------------------+------------------------------+-----------------+-------------------------------+
|ageofparticipant|clinician.branch|clinician.name|clinician.role|drug_used|experimentenddate|experimentstartdate|noofhourspassedatfirstreaction|result.conclusion|result.sideeffectsonparticipant|
+----------------+----------------+--------------+--------------+---------+-----------------+-------------------+------------------------------+-----------------+-------------------------------+
|               0|               0|             0|           109|        0|                0|                  0|                            73|               53|                              0|
+----------------+----------------+--------------+--------------+---------+-----------------+-------------------+------------------------------+-----------------+-------------------------------+



## Cleaning

1. Flatten df
2. Rename the column
3. Address null value
4. Change the date columns to the regular date

In [18]:
# 1.flatten all nested columns

columns = ['ageofparticipant',
 'clinician.branch',
 'clinician.name',
 'clinician.role',
 'drug_used',
 'experimentenddate',
 'experimentstartdate',
 'noofhourspassedatfirstreaction',
 'result.conclusion',
 'result.sideeffectsonparticipant']

flatten_drug_trial = drug_trial.select(columns) 

In [20]:
# 2 Rename the columns

new_columns ={'ageofparticipant':'participant_age',
 'branch':'clinic_branch',
 'name':'clinician_name',
 'role':'clinician_assistant',
 'drug_used': 'drug_used',
 'experimentenddate':'experiment_end_date',
 'experimentstartdate':'experiment_start_date',
 'noofhourspassedatfirstreaction':'hours_passed_at_first_reaction',
 'conclusion':'test_conclusion',
 'sideeffectsonparticipant':'observed_result'}

flatten_drug_trial = flatten_drug_trial.withColumnsRenamed(new_columns) 

In [21]:
flatten_drug_trial.show()

+---------------+-------------+--------------+-------------------+---------+-------------------+---------------------+------------------------------+---------------+---------------+
|participant_age|clinic_branch|clinician_name|clinician_assistant|drug_used|experiment_end_date|experiment_start_date|hours_passed_at_first_reaction|test_conclusion|observed_result|
+---------------+-------------+--------------+-------------------+---------+-------------------+---------------------+------------------------------+---------------+---------------+
|             19|      Ontario|          Saul|          therapist|  Placebo|      1619827200000|        1617235200000|                            52|  BP normalized| rashes on neck|
|             14|      Ontario|          Saul|              nurse| Naproxen|      1619827200000|        1617235200000|                            78|      Follow-up|            N/A|
|             17|      Ontario|          Saul|              nurse|  Placebo|      16198272

In [23]:
flatten_drug_trial.describe().show()

+-------+------------------+-------------+--------------+-------------------+---------+--------------------+---------------------+------------------------------+---------------+---------------+
|summary|   participant_age|clinic_branch|clinician_name|clinician_assistant|drug_used| experiment_end_date|experiment_start_date|hours_passed_at_first_reaction|test_conclusion|observed_result|
+-------+------------------+-------------+--------------+-------------------+---------+--------------------+---------------------+------------------------------+---------------+---------------+
|  count|              3586|         3586|          3586|               3477|     3586|                3586|                 3586|                          3513|           3533|           3586|
|   mean|17.507250418293363|         NULL|          NULL|               NULL|     NULL|1.618381578137200...| 1.615813671834913...|             44.89097637346997|           NULL|           NULL|
| stddev|2.3066401927555233|  

In [24]:
# filling the null value

flatten_drug_trial = flatten_drug_trial.na.fill({'clinician_assistant':'not provided',
                                                   'test_conclusion': 'not provided'})

In [25]:
flatten_drug_trial.show(200)

+---------------+-------------+--------------+-------------------+---------+-------------------+---------------------+------------------------------+---------------+---------------+
|participant_age|clinic_branch|clinician_name|clinician_assistant|drug_used|experiment_end_date|experiment_start_date|hours_passed_at_first_reaction|test_conclusion|observed_result|
+---------------+-------------+--------------+-------------------+---------+-------------------+---------------------+------------------------------+---------------+---------------+
|             19|      Ontario|          Saul|          therapist|  Placebo|      1619827200000|        1617235200000|                            52|  BP normalized| rashes on neck|
|             14|      Ontario|          Saul|              nurse| Naproxen|      1619827200000|        1617235200000|                            78|      Follow-up|            N/A|
|             17|      Ontario|          Saul|              nurse|  Placebo|      16198272

In [26]:
flatten_drug_trial.describe().show()

+-------+------------------+-------------+--------------+-------------------+---------+--------------------+---------------------+------------------------------+---------------+---------------+
|summary|   participant_age|clinic_branch|clinician_name|clinician_assistant|drug_used| experiment_end_date|experiment_start_date|hours_passed_at_first_reaction|test_conclusion|observed_result|
+-------+------------------+-------------+--------------+-------------------+---------+--------------------+---------------------+------------------------------+---------------+---------------+
|  count|              3586|         3586|          3586|               3586|     3586|                3586|                 3586|                          3513|           3586|           3586|
|   mean|17.507250418293363|         NULL|          NULL|               NULL|     NULL|1.618381578137200...| 1.615813671834913...|             44.89097637346997|           NULL|           NULL|
| stddev|2.3066401927555233|  